# Dimensionality Reduction And Clustering

**REQUIRED DAY 2**

## Load your checkpoint

Fresh kernel -- loading back the `adata` saved at the end of [06_normalization_and_feature_selection.ipynb](06_normalization_and_feature_selection.ipynb) (normalized, log-transformed, highly-variable genes flagged).

In [ ]:
import scanpy as sc

adata = sc.read_h5ad("results/checkpoint_06_normalized.h5ad")
adata


## PCA, neighbors, UMAP

A cell-by-gene matrix with thousands of genes is too high-dimensional to cluster or visualize directly. The standard pipeline compresses it in stages:

In [ ]:
sc.pp.pca(adata, n_comps=50)  # compute a generous number up front; you choose how many to actually use below
sc.pl.pca_variance_ratio(adata, n_pcs=50, log=True)

## How many PCs actually matter here?

`n_comps=50` above was deliberately generous — not a claim that all 50 are meaningful. The plot shows how much variance each additional PC explains; **there is no universally correct cutoff**, same as the clustering resolution below. Look for where the curve stops dropping sharply and flattens out (the "elbow") — the PCs past that point are contributing very little on top of what earlier ones already captured, mostly noise. Pick a number, then use it below.

In [ ]:
N_PCS = None  # set this based on where the curve above flattens out, then re-run
assert N_PCS is not None, "choose a value based on the variance-ratio plot above"

sc.pp.neighbors(adata, n_pcs=N_PCS)         # build a graph of each cell's nearest neighbors, using only your chosen PCs
sc.tl.umap(adata)                           # 2D layout for visualization only — not used for clustering itself
sc.pl.umap(adata)

PCA finds the axes of greatest variation; the neighbor graph captures which cells are transcriptionally similar; UMAP is a 2D projection for *looking at* that structure — it's a visualization, not the thing clustering actually operates on. Whatever `N_PCS` you chose feeds directly into the neighbor graph — too few and you lose real structure, too many and you're clustering on noise alongside signal.

## Clustering, and the parameter nobody can tell you the "right" value for

In [ ]:
import matplotlib.pyplot as plt

resolutions_to_try = [0.2, 0.4, 0.6, 0.8, 1.0, 1.2, 1.5, 2.0]
n_clusters = []
for r in resolutions_to_try:
    sc.tl.leiden(adata, resolution=r, key_added=f"leiden_scan_{r}")
    n_clusters.append(adata.obs[f"leiden_scan_{r}"].nunique())

fig, ax = plt.subplots(figsize=(5, 4))
ax.plot(resolutions_to_try, n_clusters, marker="o")
ax.set_xlabel("resolution")
ax.set_ylabel("number of clusters found")

`resolution` controls how fine-grained the clusters are — higher resolution means more, smaller clusters. **There is no universally correct resolution.** The scan above shows cluster count as a function of resolution for *your* choice of PCs — it will often rise, then plateau for a stretch (a real stability region worth noticing), then rise again. Pick one resolution from your scan — not necessarily where it plateaus, that's a reasonable default argument but not the only valid one — and use it below:

In [ ]:
RESOLUTION = None  # pick a value based on the scan above, then re-run
assert RESOLUTION is not None, "choose a value based on the resolution scan above"

sc.tl.leiden(adata, resolution=RESOLUTION, key_added="leiden")
sc.pl.umap(adata, color="leiden")

The number itself isn't defensible on its own — **the resolution you end up using should be justified against something concrete**, like whether clusters separate along known marker genes (checked in [08_cell_type_annotation.ipynb](08_cell_type_annotation.ipynb)), or roughly matches the number of cell types you'd actually expect in this sample (PBMCs: T cells, B cells, NK cells, a couple of monocyte subtypes — very roughly 5-8 broad types), not left at whatever a tutorial's default happened to be.

## The other thing a cluster boundary might mean: not biology

If today's data had multiple lanes, batches, or donors, a cluster boundary that lines up suspiciously well with one of those variables — rather than with any marker gene — is a warning sign, not a discovery: could a batch, lane, or chemistry effect explain that boundary better than a real cell-type difference? Today's single shared sample has one lane pair (L001/L002) that were combined during alignment specifically so this isn't a live confound in your output — but the check is worth running as a habit, since it will matter the moment you work with real multi-batch data.

In [ ]:
sc.pl.umap(adata, color=["leiden", "total_counts", "pct_counts_mt"])

If a cluster boundary tracks `total_counts` or `pct_counts_mt` more cleanly than it tracks any biology you'd expect, that's a QC artifact bleeding into your clustering, not a cell type.

## Save your checkpoint

[08_cell_type_annotation.ipynb](08_cell_type_annotation.ipynb) loads this back in.

In [ ]:
adata.write_h5ad("results/checkpoint_07_clustered.h5ad")
print("Saved to results/checkpoint_07_clustered.h5ad")


## Practice

State the PC count and the resolution you chose, and why, in one sentence each -- the two real judgment calls this notebook asked you to make.

## Further reading

- [Single-cell best practices — Dimensionality Reduction](https://www.sc-best-practices.org/preprocessing_visualization/dimensionality_reduction.html)
- [Single-cell best practices — Clustering](https://www.sc-best-practices.org/cellular_structure/clustering.html)